# 手写 PyTorch PPO —— 逐块运行版

本 notebook 是 `2-pytorch_ppo_simple.py` 的拆分版本（后者又是 `2-pytorch_ppo.py` 去掉 SwanLab 的版本）。
上一个 notebook 用 SB3 的 `model.learn()` 一行搞定训练，这里把那个黑盒拆开：
采样轨迹 → 算 GAE 优势 → PPO 裁剪目标更新，全部自己写。

逐块运行，看每块的输出，就能对上 PPO 论文里的每一项。

## 1. 导入依赖、选设备、固定随机种子

固定 seed 是为了让每次运行结果可复现——手写 RL 调试时这一步很关键。

In [1]:
import os
import random
import sys
from pathlib import Path

# notebook 没有 __file__，从 cwd 推出 code/ 目录才能 import device_utils
_CODE_ROOT = Path.cwd().parent if Path.cwd().name == "chapter01_cartpole" else Path.cwd()
if str(_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(_CODE_ROOT))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gymnasium as gym

from device_utils import describe_device, print_device_report, resolve_torch_device

# 脚本里的命令行参数，在 notebook 里提成常量，直接改这里
SEED = 42
ITERATIONS = 10          # 脚本默认 40；先用小值跑通，再调大
STEPS_PER_ROLLOUT = 2048
GUI = False              # 弹窗会阻塞 kernel，notebook 里保持 False

device = resolve_torch_device("auto")   # 也可以写 "cpu" / "mps" / "cuda"
print_device_report(device)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PyTorch device report
  PyTorch:  2.12.0
  CUDA:     False
  MPS:      True
  Selected: Apple MPS (Metal) (mps)


## 2. Actor-Critic 网络

两个独立的 MLP：Actor 输出动作 logits，Critic 输出状态价值 V(s)。
注意正交初始化——Actor 输出层用 `gain=0.01`，让初始策略接近均匀分布，避免一开始就偏向某个动作。
运行后看看打印出的网络结构。

In [2]:
class ActorCritic(nn.Module):
    """
    Separate Actor-Critic networks (matching SB3's MlpPolicy):
    - Actor and Critic use their own hidden layers, avoiding gradient interference
    - Orthogonal init: gain=0.01 on the actor output layer keeps the initial policy near-uniform
    """

    def __init__(self, obs_dim=4, act_dim=2, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
        self._init_weights()

    def _init_weights(self):
        """Orthogonal initialization, matching SB3's default"""
        for module in self.actor:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        for module in self.critic:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
                nn.init.constant_(module.bias, 0)
        # Use a small gain on the actor's output layer -> initial policy close to uniform
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.constant_(self.actor[-1].bias, 0)
        # Critic output layer gain=1
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)
        nn.init.constant_(self.critic[-1].bias, 0)

    def forward(self, x):
        logits = self.actor(x)
        value = self.critic(x)
        return logits, value.squeeze(-1)

    def get_action(self, obs, deterministic=False):
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits=logits)
        if deterministic:
            action = logits.argmax(dim=-1)
        else:
            action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob, value

ActorCritic()

ActorCritic(
  (actor): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
  (critic): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

### 2.1 正交初始化：`nn.init.orthogonal_` 到底做了什么

上面 `_init_weights` 里那两个循环，把每个 `nn.Linear` 的权重换成了**正交矩阵**：
生成的 W 满足 $W^\top W = g^2 I$，也就是**所有奇异值都等于 `gain`**。
PyTorch 的做法是先采一个高斯随机矩阵，做 QR 分解取正交因子 Q，再整体乘上 `gain`。

奇异值决定了这一层对信号的**拉伸倍率**：

- 奇异值全相等 → 这个线性变换在所有方向上等比缩放，不挑方向；
- 前向时输入范数乘以 `gain` 传给下一层，反向时梯度乘的是 $W^\top$（奇异值同样全是 `gain`），
  所以信号和梯度在层间传播既不爆炸也不消失。

下面这块对比正交初始化和普通高斯初始化的奇异值分布，看 min/max 差多少。

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

W = torch.empty(64, 64)
nn.init.orthogonal_(W, gain=np.sqrt(2))          # 和网络隐藏层用的一样
print(f"正交初始化 gain=√2 : 奇异值 min={W.svd().S.min():.5f}  max={W.svd().S.max():.5f}")

G = torch.empty(64, 64)
nn.init.normal_(G, std=0.1)                       # 对照组：普通高斯初始化
print(f"普通高斯初始化     : 奇异值 min={G.svd().S.min():.5f}  max={G.svd().S.max():.5f}")
print(f"  → 高斯的最大/最小奇异值相差 {(G.svd().S.max()/G.svd().S.min()):.0f} 倍")

# 正交层对输入范数做的事：整体乘以 gain，仅此而已
x = torch.randn(1000, 64)
ratio = (W @ x.T).norm(dim=0).mean() / x.norm(dim=1).mean()
print(f"\n正交层 输出范数/输入范数 = {ratio:.5f}  (= √2 = {np.sqrt(2):.5f})")

**为什么 gain 用 √2？** 补偿激活函数吃掉的方差。以 ReLU 为例，它把一半输入置零，
输出方差大约减半，权重放大 √2 倍正好补回来（这就是 He 初始化的由来）。

严格来说这里的激活是 **Tanh**，PyTorch 给 Tanh 推荐的 gain 是 5/3；写 √2 是**照抄 SB3 的默认值**
（SB3 又继承自 OpenAI Baselines），属于跑得通的经验惯例而非理论最优。
目的是让这份手写实现和第一份 SB3 版行为可比。

`nn.init.constant_(module.bias, 0)`：偏置置零，不给任何神经元预设的激活倾向。

### 2.2 真正要紧的是被覆盖掉的输出层

两个循环把 **所有** Linear 层都设成了 `gain=√2`，紧接着的四行又把输出层单独改掉——
**这是故意写两遍**，不是冗余：

```python
nn.init.orthogonal_(self.actor[-1].weight,  gain=0.01)   # 覆盖
nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)    # 覆盖
```

Actor 输出层的 `gain=0.01` 是 PPO 里最关键的一处。下面这块对比两种 gain 下的初始动作分布。

In [ ]:
def actor_head_stats(gain):
    """造一个 actor 输出层，看它给出的初始动作分布有多均匀"""
    head = nn.Linear(64, 2)
    nn.init.orthogonal_(head.weight, gain=gain)
    nn.init.constant_(head.bias, 0)
    hidden = torch.tanh(torch.randn(2000, 64))    # 模拟隐藏层输出
    logits = head(hidden)
    probs = logits.softmax(-1)
    print(f"gain={gain:<6.3f} |logits|均值={logits.abs().mean():.4f}  "
          f"两个动作平均概率={probs.mean(0).tolist()}  最大概率均值={probs.max(-1).values.mean():.4f}")

torch.manual_seed(0)
actor_head_stats(np.sqrt(2))   # 如果输出层没被覆盖，会是这样
actor_head_stats(0.01)         # 实际代码用的

# 看真实模型：输出层权重比隐藏层小两个数量级
model_demo = ActorCritic()
print(f"\n隐藏层权重绝对值均值     : {model_demo.actor[0].weight.abs().mean():.4f}")
print(f"Actor 输出层权重绝对值均值 : {model_demo.actor[-1].weight.abs().mean():.4f}")
print(f"Critic 输出层权重绝对值均值: {model_demo.critic[-1].weight.abs().mean():.4f}")

`gain=0.01` 把 logits 压到接近 0，softmax 后两个动作各约 50%，即**初始策略是最大熵的**。
这对 PPO 有两层意义：

1. **探索**：训练一开始就充分探索，不会因为随机初始化恰好偏向"总是向左推"而卡在坏策略里。
2. **更新稳定**：PPO 的裁剪目标算的是 $r = \exp(\log\pi_{new} - \log\pi_{old})$。
   初始策略若已经很尖锐（某个动作 99%），第一次更新的 ratio 很容易冲出 $[1-\epsilon,\, 1+\epsilon]$，
   梯度直接被裁没、或者 KL 炸掉。均匀的初始策略让前几轮更新平稳。
   （第 7 节训练循环里打印的 `KL` 和 `clip%` 就是在盯这件事。）

Critic 输出层 `gain=1.0`：它回归的是真实回报值（CartPole 里能到几十上百），
保持单位缩放即可，不需要额外放大或抑制。

## 3. 采样轨迹（Rollout）

跑 `num_steps` 步，把每一步的 obs / action / log_prob / value / reward 存下来。
关键区分：`terminated`（杆子倒了，V(s')=0）和 `truncated`（到达步数上限，V(s') 仍要 bootstrap）。
这个区分搞错，价值函数就会系统性偏低。

In [3]:
def collect_rollout(
    model,
    env,
    obs,
    device,
    episode_reward=0.0,
    episode_length=0,
    num_steps=2048,
):
    """
    Collect a trajectory, correctly handling terminated vs truncated:
    - terminated (the pole fell over): V(s')=0
    - truncated (hit the step limit): V(s') needs to be bootstrapped
    - end of rollout without termination: also bootstrap with V(s')
    """
    transitions = []
    completed_rewards = []
    completed_lengths = []

    for _ in range(num_steps):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, log_prob, value = model.get_action(obs_tensor)

        next_obs, reward, terminated, truncated, _ = env.step(action.item())
        with torch.no_grad():
            if terminated:
                next_value = 0.0
            else:
                next_obs_tensor = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
                _, next_value_tensor = model(next_obs_tensor)
                next_value = next_value_tensor.item()

        # Store this step's V(s') so termination, truncation, and end-of-rollout all share one GAE formula.
        transitions.append({
            "obs": obs,
            "action": action.item(),
            "log_prob": log_prob.item(),
            "value": value.item(),
            "reward": float(reward),
            "terminated": terminated,
            "truncated": truncated,
            "next_value": next_value,
        })

        episode_reward += float(reward)
        episode_length += 1

        obs = next_obs
        if terminated or truncated:
            completed_rewards.append(episode_reward)
            completed_lengths.append(episode_length)
            episode_reward = 0.0
            episode_length = 0
            obs, _ = env.reset()

    return (
        transitions,
        obs,
        completed_rewards,
        completed_lengths,
        episode_reward,
        episode_length,
    )

## 4. 计算 GAE 优势

倒序递推 `gae = delta + gamma * lam * (1 - episode_end) * gae`。
`returns = 优势 + value` 是 Critic 的回归目标（不归一化）；
`advantages` 做了标准化，只给策略损失用。

In [4]:
def compute_gae(transitions, gamma=0.99, lam=0.95):
    """
    Generalized Advantage Estimation, handling correctly:
    - terminated (a genuine episode end): do not propagate GAE, V(s')=0
    - truncated (time limit): bootstrap with V(s'), but do not propagate GAE across the reset
    - end of rollout: bootstrap with the stored V(s')
    """
    raw_advantages = []
    gae = 0

    for step in reversed(range(len(transitions))):
        t = transitions[step]
        episode_end = t["terminated"] or t["truncated"]
        delta = t["reward"] + gamma * t["next_value"] - t["value"]
        gae = delta + gamma * lam * (1.0 - float(episode_end)) * gae
        raw_advantages.insert(0, gae)

    raw_advantages = torch.tensor(raw_advantages, dtype=torch.float32)
    values = torch.tensor([t["value"] for t in transitions], dtype=torch.float32)
    # The Critic learns the unnormalized return target; normalization is only for the policy loss.
    returns = raw_advantages + values
    advantages = (raw_advantages - raw_advantages.mean()) / (
        raw_advantages.std(unbiased=False) + 1e-8
    )

    return advantages, returns

## 5. PPO 更新

核心就是这三行裁剪目标：

```python
ratio = exp(new_log_prob - old_log_prob)
surr1, surr2 = ratio * A, clamp(ratio, 1-eps, 1+eps) * A
policy_loss = -min(surr1, surr2).mean()
```

同一批数据反复用 `epochs=10` 轮，靠 clip 限制策略别更新太远。
返回的 `approx_kl` 和 `clip_fraction` 是判断更新幅度是否合理的关键指标。

In [5]:
def ppo_update(model, optimizer, transitions, advantages, returns, device,
               clip_eps=0.2, epochs=10, batch_size=64):
    """PPO clipped objective update"""
    obs = np.array([t["obs"] for t in transitions])
    actions = np.array([t["action"] for t in transitions])
    old_log_probs = np.array([t["log_prob"] for t in transitions])

    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
    actions = torch.as_tensor(actions, dtype=torch.long, device=device)
    old_log_probs = torch.as_tensor(old_log_probs, dtype=torch.float32, device=device)
    advantages = advantages.to(device)
    returns = returns.to(device)

    total_policy_loss = 0
    total_value_loss = 0
    total_entropy = 0
    total_kl = 0
    total_clip_frac = 0
    n_updates = 0

    for _ in range(epochs):
        indices = np.random.permutation(len(transitions))

        for start in range(0, len(transitions), batch_size):
            idx = indices[start:start + batch_size]

            batch_obs = obs[idx]
            batch_actions = actions[idx]
            batch_old_log_probs = old_log_probs[idx]
            batch_advantages = advantages[idx]
            batch_returns = returns[idx]

            logits, values = model(batch_obs)
            dist = torch.distributions.Categorical(logits=logits)
            new_log_probs = dist.log_prob(batch_actions)

            # PPO clipped objective
            ratio = torch.exp(new_log_probs - batch_old_log_probs)
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()

            # Value function loss
            value_loss = ((values - batch_returns) ** 2).mean()

            # Entropy bonus (encourages exploration)
            entropy = dist.entropy().mean()

            loss = policy_loss + 0.5 * value_loss - 0.0 * entropy

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            # Track metrics
            with torch.no_grad():
                log_ratio = new_log_probs - batch_old_log_probs
                # Non-negative KL approximation, matching SB3's approx_kl computation.
                total_kl += ((log_ratio.exp() - 1) - log_ratio).mean().item()
                total_clip_frac += ((ratio - 1.0).abs() > clip_eps).float().mean().item()

            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()
            total_entropy += entropy.item()
            n_updates += 1

    return {
        "policy_loss": total_policy_loss / n_updates,
        "value_loss": total_value_loss / n_updates,
        "entropy": total_entropy / n_updates,
        "approx_kl": total_kl / n_updates,
        "clip_fraction": total_clip_frac / n_updates,
    }

## 6. 建环境，看看它长什么样

In [6]:
env = gym.make("CartPole-v1")
env.action_space.seed(SEED)
obs, _ = env.reset(seed=SEED)

print("=" * 50)
print("CartPole-v1 environment info")
print("=" * 50)
print(f"  Observation space:  {env.observation_space}")
print(f"  Action space:  {env.action_space}")
print(f"  Observation upper bound:  {env.observation_space.high}")
print(f"  Observation lower bound:  {env.observation_space.low}")
print(f"  Termination condition:  position > ±{env.unwrapped.x_threshold}, "
      f"angle > ±{env.unwrapped.theta_threshold_radians:.4f} rad "
      f"(≈ ±{np.degrees(env.unwrapped.theta_threshold_radians):.0f}°)")
print("=" * 50)

model = ActorCritic().to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

CartPole-v1 environment info
  Observation space:  Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  Action space:  Discrete(2)
  Observation upper bound:  [4.8               inf 0.41887903        inf]
  Observation lower bound:  [-4.8               -inf -0.41887903        -inf]
  Termination condition:  position > ±2.4, angle > ±0.2094 rad (≈ ±12°)


## 7. 训练循环

每一轮：采样 → 算 GAE → 线性衰减学习率 → PPO 更新 → 打印指标。
重点看 `Mean reward` 是否上升、`KL` 是否稳定在 0.01 上下（太大说明步子迈得过猛）。

这一块会阻塞 kernel 几十秒到几分钟，取决于上面的 `ITERATIONS`。

In [7]:
print(f"Starting training on {describe_device(device)} (pure PyTorch PPO)...")
print("-" * 60)

total_timesteps = 0
metric_rows = []
ongoing_episode_reward = 0.0
ongoing_episode_length = 0

for iteration in range(ITERATIONS):
    # 采样
    (
        transitions,
        obs,
        ep_rewards,
        ep_lengths,
        ongoing_episode_reward,
        ongoing_episode_length,
    ) = collect_rollout(
        model, env, obs, device,
        ongoing_episode_reward, ongoing_episode_length, STEPS_PER_ROLLOUT,
    )
    total_timesteps += len(transitions)

    # 优势 + Critic 回归目标
    advantages, returns = compute_gae(transitions)

    # 学习率线性衰减
    frac = 1.0 - iteration / ITERATIONS
    lr = 3e-4 * frac
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # PPO 更新
    metrics = ppo_update(model, optimizer, transitions, advantages, returns, device)

    # explained variance：采样时的价值预测 vs 真实回报目标
    return_values = returns.numpy()
    rollout_values = np.array([t["value"] for t in transitions])
    var_returns = np.var(return_values)
    if var_returns < 1e-6:
        explained_variance = 0.0
    else:
        explained_variance = 1 - np.var(return_values - rollout_values) / var_returns

    mean_reward = np.mean(ep_rewards) if ep_rewards else 0
    mean_ep_len = np.mean(ep_lengths) if ep_lengths else 0

    metric_rows.append({
        "seed": SEED,
        "iteration": iteration + 1,
        "total_timesteps": total_timesteps,
        "completed_episodes": len(ep_rewards),
        "mean_episode_reward": mean_reward,
        "mean_episode_length": mean_ep_len,
        "policy_loss": metrics["policy_loss"],
        "value_loss": metrics["value_loss"],
        "entropy": metrics["entropy"],
        "approx_kl": metrics["approx_kl"],
        "clip_fraction": metrics["clip_fraction"],
        "explained_variance": explained_variance,
        "learning_rate": lr,
    })

    print(
        f"  Iteration {iteration + 1:2d}/{ITERATIONS} | "
        f"Episodes: {len(ep_rewards):3d} | "
        f"Mean reward: {mean_reward:6.1f} | "
        f"KL: {metrics['approx_kl']:.4f} | "
        f"clip%: {metrics['clip_fraction']:.1%}"
    )

print("-" * 60)

Starting training on Apple MPS (Metal) (pure PyTorch PPO)...
------------------------------------------------------------
  Iteration  1/10 | Episodes:  89 | Mean reward:   22.8 | KL: 0.0086 | clip%: 13.5%
  Iteration  2/10 | Episodes:  68 | Mean reward:   29.6 | KL: 0.0075 | clip%: 8.4%
  Iteration  3/10 | Episodes:  45 | Mean reward:   45.8 | KL: 0.0057 | clip%: 6.8%
  Iteration  4/10 | Episodes:  29 | Mean reward:   70.2 | KL: 0.0052 | clip%: 7.2%
  Iteration  5/10 | Episodes:  18 | Mean reward:  107.7 | KL: 0.0056 | clip%: 8.5%
  Iteration  6/10 | Episodes:  12 | Mean reward:  175.4 | KL: 0.0038 | clip%: 3.3%
  Iteration  7/10 | Episodes:  10 | Mean reward:  195.0 | KL: 0.0049 | clip%: 6.4%
  Iteration  8/10 | Episodes:   8 | Mean reward:  267.1 | KL: 0.0028 | clip%: 2.3%
  Iteration  9/10 | Episodes:   9 | Mean reward:  226.8 | KL: 0.0016 | clip%: 0.5%
  Iteration 10/10 | Episodes:   6 | Mean reward:  356.0 | KL: 0.0005 | clip%: 0.0%
-----------------------------------------------

## 8. 保存指标 CSV 和模型

CSV 可以用同目录的 `plot_curves.py` 画曲线。

In [8]:
import csv

os.makedirs("output", exist_ok=True)
log_csv = "output/training_metrics.csv"
model_path = "output/pytorch_ppo_cartpole.pth"

with open(log_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(metric_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metric_rows)
print(f"Raw training metrics saved to {log_csv}")

torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Raw training metrics saved to output/training_metrics.csv
Model saved to output/pytorch_ppo_cartpole.pth


## 9. 评估

用确定性策略（取 argmax 而不是采样）跑 20 个回合。CartPole-v1 满分 500。

In [9]:
eval_rewards = []
for _ in range(20):
    obs, _ = env.reset(seed=SEED + 10_000 + len(eval_rewards))
    done, truncated, score = False, False, 0
    while not (done or truncated):
        obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            action, _, _ = model.get_action(obs_tensor, deterministic=True)
        obs, reward, done, truncated, _ = env.step(action.item())
        score += reward
    eval_rewards.append(score)

print(f"Training complete! 20-episode evaluation: {np.mean(eval_rewards):.1f} +/- {np.std(eval_rewards):.1f}")
env.close()

Training complete! 20-episode evaluation: 494.4 +/- 14.3


## 10. 演示（可选弹窗）

把上面的 `GUI = True` 再运行这块会弹出动画窗口；无显示器环境下会自动跳过。

In [10]:
if GUI:
    vis_env = gym.make("CartPole-v1", render_mode="human")
    print("Demoing what the agent learned (5 episodes)...")
    for ep in range(5):
        obs, _ = vis_env.reset(seed=SEED + 20_000 + ep)
        done, truncated, score = False, False, 0
        while not (done or truncated):
            obs_tensor = torch.as_tensor(obs, dtype=torch.float32, device=device)
            with torch.no_grad():
                action, _, _ = model.get_action(obs_tensor, deterministic=True)
            obs, reward, done, truncated, _ = vis_env.step(action.item())
            score += reward
        print(f"  Demo episode {ep + 1} score: {score}")
    vis_env.close()
else:
    print("GUI = False，只跑评估不弹窗。想看动画就把上面的 GUI 改成 True。")

GUI = False，只跑评估不弹窗。想看动画就把上面的 GUI 改成 True。


## 11. 拆开看一次 rollout 里到底存了什么

脚本里这些张量一闪而过。这块只采样 5 步，把 transition 字典逐条打印出来，
对照第 3、4 节的公式看 `value` / `next_value` / `reward` 是怎么进 GAE 的。

In [11]:
demo_env = gym.make("CartPole-v1")
demo_obs, _ = demo_env.reset(seed=0)

transitions_demo, *_ = collect_rollout(model, demo_env, demo_obs, device, num_steps=5)
for i, t in enumerate(transitions_demo):
    print(f"step {i}: action={t['action']}  reward={t['reward']}  "
          f"value={t['value']:.3f}  next_value={t['next_value']:.3f}  "
          f"terminated={t['terminated']}  truncated={t['truncated']}")

adv_demo, ret_demo = compute_gae(transitions_demo)
print("\nadvantages:", np.round(adv_demo.numpy(), 3))
print("returns:   ", np.round(ret_demo.numpy(), 3))
demo_env.close()

step 0: action=1  reward=1.0  value=44.287  next_value=44.211  terminated=False  truncated=False
step 1: action=0  reward=1.0  value=44.211  next_value=44.275  terminated=False  truncated=False
step 2: action=0  reward=1.0  value=44.275  next_value=44.304  terminated=False  truncated=False
step 3: action=1  reward=1.0  value=44.304  next_value=44.264  terminated=False  truncated=False
step 4: action=0  reward=1.0  value=44.264  next_value=44.300  terminated=False  truncated=False

advantages: [ 1.326  0.805  0.035 -0.73  -1.436]
returns:    [46.766 46.334 45.872 45.379 44.857]
